<a href="https://colab.research.google.com/github/lalawee/special-octo-barnacle/blob/main/GridWorld_MC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Monte Carlo First Visit Method to solve a simple GridWorld problem.#
Goal is to estimate the state-value function V(s) for a given policy.
<br>In this case, we are using a RANDOM UNIFORM POLICY to train.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys

# where gridworld.py is located
# modify path accordingly
sys.path.append('/content/drive/MyDrive')


In [ ]:
import numpy as np
import gridworld

#**Main starts here**

**Standard Grid:**<br>
x means you can't go there<br>
s means start position<br>
number means reward at that state<br>
**.  .  .  1**<br>
**.  x  . -1**<br>
**s  .  .  .**<br>





In [ ]:
# define a grid that describes the reward for arriving at each state and possible actions at each state
# the grid looks like this:
#
# x means you can't go there
# s means start position
# number means reward at that state
#   0  1  2  3
# 0 .  .  .  1
# 1 .  x  . -1
# 2 s  .  .  .
#
# We go by (row, col)
#
MAX_EPISODES = 10000
GAMMA = 0.9
CHECKPOINTS = [100, 500, 1000, 5000, 10000]
env = gridworld.standard_grid() #3x4 grid
returns = {}  # Dictionary to store returns for each state, returns[s] points to a list
state_values = {} # Dictionary to store state value functions

In [ ]:
#
# Generate Episodes
#
for ep in range(MAX_EPISODES):
	state = env.reset() #return a tuple, state at start
	trajectory = [] #to store trajectory of 1 episode

  #
	# generate 1 episode
  #
	while True:
		# using UNIFORM RANDOM as a policy to train (behavioral policy)
		action = np.random.choice(['U','D','L','R'])

		next_state, reward, done, invalid = env.step(action)
		if invalid: continue #go back and get another random action

		trajectory.append((state, reward)) #St, Rt+1
		state = next_state
		if done: break

  #
	# Analyse the 1 episode generated
	# E.g. trajectory = [((2, 0), 0), ((2, 1), 0), ((2, 0), 0), ((2, 1), 0), ((2, 2), 0), ((1, 2), -1)]
	#
	reverse_trajectory = trajectory[::-1]
	reverse_trajectory_states = list(map(lambda x:x[0], reverse_trajectory)) # just the states only
	G = 0

	#
	# GO THRU trajectory BACKWARD to CALC averages
	# trajectory[] contains (state, reward) of only 1 episode
	#
	for idx, step in enumerate(reverse_trajectory):
		s, r = step
		G = GAMMA*G + r # say, after 3 steps, G=GAMMA*(GAMMA*(GAMMA*G + r3) + r2) + r1
                    #                     G=GAMMA^3*G + GAMMA^2*r3 + GAMMA*r2 + r1
                    #                     G=GAMMA^2*r3 + GAMMA*r2 + r1

    #
    # First-visit MC: only update the FIRST OCCURENCE of the state
    #                         5       4       3       2       1       0
		# E.g. for trajectory = [(2, 0), (2, 1), (2, 0), (2, 1), (2, 2), (1, 2)]
		#      it will be idx: 0, 1, 4, 5
		#      Cause for example when idx=2, state is (2,1), but this state is in reverse_trajectory_states[3:],
		#      reverse_trajectory_states[3:] means [(2,0),(2,1),(2,0)], so we will ignore its processing.
		#      Hence, we only consider the state (2,1) from the first occurence counting from the start of trajectory.
		#
		if s not in reverse_trajectory_states[idx+1:]:
			# we add state s to dict returns, returns[s] points to a list
			if s in returns: returns[s].append(G) # returns is a dict that points to lists
			else:            returns[s] = [G] # first guy in the list

			# update state_values[s] dict, which stores mean gain results of each state s
			state_values[s] = np.mean(returns[s]) # returns[s] points to a list

    # The terminal states will have 0 value cause no further rewards are received after reaching them.
    # That is, agent doesn't move from terminal states to get further rewards. That is why we
    # manually insert the 2 final values (rewards) in the terminal states in the state_values dict.
		state_values[(0,3)]=1
		state_values[(1,3)]=-1

print("Training complete.")

Training complete.


###Note: This is an ON POLICY RL. In the example, the random policy is both the behavior policy (used to generate episodes) and the target policy (being evaluated or updated). There is no separation between the behavior policy and the target policy. The values V(s) represent the expected return when following say, random policy and the value estimates are directly tied to the same random policy being followed. We didn't further alter the values by weighting them during updates, for example.


In [ ]:
print("State Values:")
gridworld.print_values(state_values,env)

State Values:
---------------------------
 0.05| 0.13| 0.26| 1.00|
---------------------------
-0.03| 0.00|-0.36|-1.00|
---------------------------
-0.11|-0.22|-0.38|-0.67|
